In [39]:
import constants as c
import numpy as np
import math
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy.integrate import solve_ivp

vk = c.vk
vrf = c.vrf
z_offset = c.z0
xy1k = c.xy1k
xy2k = c.xy2k
pi = math.pi
um = c.um
cm = c.cm
mm = c.mm
K = c.K
e = c.e

#eps = 1
alpha = 1/137.06
d_chain = 3.7066438742e-06  # 3.7 µm separation between ions
delta_ion_2 = 0#-4.9903032463e-14 # small displacement of ion 2
#m_dm = 1e-27
m_ion = c.m 
Z_ion = c.Z
freq_x = 719430.7131391969
freq_y = 3031200.0099101723
freq_z = 3002153.5607483205
omega_vec = np.asarray(2*pi*np.array([freq_x,freq_y,freq_z]), dtype=float) 
hbar = 1.0545718e-34 
mode_y = 2*pi*np.array([3031200.01,2944587.06,2818861.50])
mode_z = 2*pi*np.array([3002153.56,2914677.59,2787603.39])

In [40]:
"""
xyz : ndarray, shape (n, 3)
        Particle positions.
"""
# RF pseudo-force
def FRF(xyz,m,Z,q=c.e,omrf=c.omega,VRF=c.vrf,ymin=c.y11,yedge1=c.y21,yedge2=c.y12,ymax=c.y22):
    xyz = np.asarray(xyz, dtype=float)
    if xyz.ndim != 2 or xyz.shape[1] != 3:
        raise ValueError("xyz must have shape (n, 3)")
    x = xyz[:, 0]
    y = xyz[:, 1]
    z = xyz[:, 2]
    m = float(np.asarray(m).ravel()[0])
    Z = float(np.asarray(Z).ravel()[0])
    q = float(np.asarray(q).ravel()[0])
    omrf = float(np.asarray(omrf).ravel()[0])
    VRF = float(np.asarray(VRF).ravel()[0])
    #this is the pseudo potential, which is Z^2*(Div[PhiRF]/cos(om*t))^2/(4m*omega^2)
    divypart=divatan(z,yedge2-y)-divatan(z,yedge1-y)+divatan(z,ymin-y)-divatan(z,ymax-y)
    divzpart=divatan(yedge2-y,z)-divatan(yedge1-y,z)+divatan(ymin-y,z)-divatan(ymax-y,z) 
    divyparty=-2*divypart*(divatandown(z,yedge2-y)-divatandown(z,yedge1-y)+divatandown(z,ymin-y)-divatandown(z,ymax-y)) 
    divypartz=2*divypart*(divatanup(z,yedge2-y)-divatanup(z,yedge1-y)+divatanup(z,ymin-y)-divatanup(z,ymax-y)) 
    divzparty=-2*divzpart*(divatanup(yedge2-y,z)-divatanup(yedge1-y,z)+divatanup(ymin-y,z)-divatanup(ymax-y,z)) 
    divzpartz=2*divzpart*(divatandown(yedge2-y,z)-divatandown(yedge1-y,z)+divatandown(ymin-y,z)-divatandown(ymax-y,z)) 
    return -1*((VRF*Z*q)/(2*pi*np.sqrt(m)*omrf))**2*np.column_stack([divypartz*0, divzparty+divyparty, divzpartz+divypartz])

# DC force
def divatan(up,down):
    #This is d(arctan2(up,down))/ddown up to a minus sign. It's useful for the pseudo-potential
    return up/(up**2+down**2)
def divatanup(up,down):
    #This is d(divatan(up,down))/dup
    return (down**2-up**2)/(up**2+down**2)**2
def divatandown(up,down):
    #This is d(divatan(up,down))/ddown
    return -2*up*down/(up**2+down**2)**2

def anatangrad(xi,yi,xyz,v): # gradient term of DC potential
    xyz = np.asarray(xyz, dtype=float)
    x = xyz[:, 0]
    y = xyz[:, 1]
    z = xyz[:, 2]
    dy=y-yi
    dx=x-xi
    r = np.sqrt(dx**2+dy**2+z**2); # added distance
    dry2=z**2+dy**2
    drx2=z**2+dx**2
    divy=z*dx/(r*dry2); # divide by factor r
    divz=-dy*dx*(1/dry2+1/drx2)/r; # divide by factor r
    divx=z*dy/(r*drx2); # divide by factor r
    return (v/(2*np.pi))*np.column_stack([divx, divy, divz])

def FDC_single(x1,y1,x2,y2,xyz,v,Z,q=c.e):
    return -Z*q * (anatangrad(x2,y2,xyz,v)-anatangrad(x2,y1,xyz,v)-anatangrad(x1,y2,xyz,v)+anatangrad(x1,y1,xyz,v))

def FDC(xyz, m, Z, q=c.e, omrf=c.omega,
        ymin=c.y11, yedge1=c.y21, yedge2=c.y12, ymax=c.y22):
    xyz = np.asarray(xyz, dtype=float)
    if xyz.ndim != 2 or xyz.shape[1] != 3:
        raise ValueError("xyz must have shape (n, 3)")
    z_original = xyz[:, 2].copy()
    force = np.zeros_like(xyz)
    for k in range(0, 40):
        xyz_k = xyz.copy()
        if (k != 19 and k != 39):
            xyz_k[:, 2] = z_original - z_offset  # keep scalar math
        else:
            xyz_k[:, 2] = z_original
        (x1k, y1k) = xy1k[k]
        (x2k, y2k) = xy2k[k]
        # enforce scalar on vk[k]
        v_k = float(np.asarray(vk[k]).ravel()[0])
        force_single = FDC_single(x1k, y1k, x2k, y2k, xyz_k, v_k, Z, q=q)
        # check shape of force_single 
        force_single = np.asarray(force_single, dtype=float)
        if force_single.shape != xyz.shape:
            raise ValueError(
                f"FDC_single returned unexpected shape {force_single.shape}; "
                f"expected {xyz.shape} at k={k}"
            )
        force += force_single
    return force

# Full Simulation Code

In [41]:
def run_simulation_many_dm_single_ion(
    x0_dms, v0_dms, x0_ion, v0_ion,
    t_span, dt, m_dm, eps, run_rutherford,
    rtol=1e-13, atol=1e-16
):
    """
    Run a single-ion / many-DM trajectory simulation.

    This version includes:
        - one trapped ion near the origin
        - many DM particles
        - ion-DM Coulomb forces
        - no DM-DM Coulomb forces

    Parameters
    ----------
    x0_dms : array-like, shape (n_dm, 3)
        Initial positions of the DM particles.

    v0_dms : array-like, shape (n_dm, 3)
        Initial velocities of the DM particles.

    x0_ion : array-like, shape (3,)
        Initial position of the single ion.

    v0_ion : array-like, shape (3,)
        Initial velocity of the single ion.

    t_span : tuple (t_start, t_end)
        Start and end time of the integration.

    dt : float
        Time increment used to build the evaluation grid.

    m_dm : float
        Mass of DM particles in kg.
    
    eps : float
        Charge of DM particles in units of elementary charge.

    run_rutherford : bool
        If True, set trap fields to zero for Rutherford scattering comparison.

    Returns
    -------
    t_eval : ndarray, shape (N,)
        Times at which the solution was evaluated.

    x_ion_sol : ndarray, shape (3, N)
        Ion position trajectory.

    v_ion_sol : ndarray, shape (3, N)
        Ion velocity trajectory.

    x_dms_sol : ndarray, shape (n_dm, 3, N)
        DM position trajectories.

    v_dms_sol : ndarray, shape (n_dm, 3, N)
        DM velocity trajectories.
    """

    x0_dms = np.asarray(x0_dms, dtype=float)
    v0_dms = np.asarray(v0_dms, dtype=float)
    x0_ion = np.asarray(x0_ion, dtype=float)
    v0_ion = np.asarray(v0_ion, dtype=float)

    if x0_dms.ndim != 2 or x0_dms.shape[1] != 3:
        raise ValueError("x0_dms must have shape (n_dm, 3)")

    if v0_dms.shape != x0_dms.shape:
        raise ValueError("v0_dms must have the same shape as x0_dms")

    if x0_ion.shape != (3,):
        raise ValueError("x0_ion must have shape (3,)")

    if v0_ion.shape != (3,):
        raise ValueError("v0_ion must have shape (3,)")

    n_dm = x0_dms.shape[0]

    # The single ion is trapped about the origin.
    r_ion_eq = np.zeros(3, dtype=float)

    def rhs_unified(t, U, r_ion_eq, use_harmonic_ion=True):
        """
        State vector layout:

        U = [
            x_ion(3),
            x_dms(3*n_dm),
            v_ion(3),
            v_dms(3*n_dm)
        ]
        """

        idx0 = 0
        idx1 = idx0 + 3
        idx2 = idx1 + 3 * n_dm
        idx3 = idx2 + 3
        idx4 = idx3 + 3 * n_dm

        x_ion = U[idx0:idx1]
        x_dms = U[idx1:idx2].reshape((n_dm, 3))

        v_ion = U[idx2:idx3]
        v_dms = U[idx3:idx4].reshape((n_dm, 3))

        dx_iondt = v_ion
        dx_dmsdt = v_dms

        dv_iondt = np.zeros(3)
        dv_dmsdt = np.zeros((n_dm, 3))

        # --------------------------------------------------
        # Trap force acting on ion
        # --------------------------------------------------
        if use_harmonic_ion:
            if run_rutherford:
                dv_iondt += np.zeros(3)
            else:
                dv_iondt += -omega_vec**2 * (x_ion - r_ion_eq)
        else:
            if not run_rutherford:
                x_ion_arr = x_ion[None, :]
                F_dc_ion = FDC(x_ion_arr, m=m_ion, Z=Z_ion)[0]
                F_rf_ion = FRF(x_ion_arr, m=m_ion, Z=Z_ion)[0]
                F_ion = F_dc_ion + F_rf_ion
                dv_iondt += F_ion / m_ion

        # --------------------------------------------------
        # Trap force acting on each DM particle
        # --------------------------------------------------
        if not run_rutherford:
            F_dc_dms = FDC(x_dms, m_dm, eps)
            F_rf_dms = FRF(x_dms, m_dm, eps)

            dv_dmsdt += (F_dc_dms + F_rf_dms) / m_dm

        # --------------------------------------------------
        # Coulomb interactions: ion <-> each DM particle
        #
        # No DM-DM Coulomb forces are included.
        # --------------------------------------------------
        delta = x_ion[None, :] - x_dms
        r = np.linalg.norm(delta, axis=1)

        forces = K * Z_ion * eps * e**2 * delta / r[:, None]**3

        # Force on ion due to DM k
        dv_iondt += forces.sum(axis=0) / m_ion

        # Equal and opposite force on DM k due to ion
        dv_dmsdt -= forces / m_dm

        return np.concatenate([
            dx_iondt,
            dx_dmsdt.flatten(),
            dv_iondt,
            dv_dmsdt.flatten()
        ])

    print("Number of DM particles:", n_dm)
    print("Initial ion position:", x0_ion)
    print("Initial ion velocity:", v0_ion)

    # Initial state
    U0 = np.concatenate([
        x0_ion,
        x0_dms.flatten(),
        v0_ion,
        v0_dms.flatten()
    ]).astype(float)

    # Time grid
    t_eval = np.arange(t_span[0], t_span[1], dt)

    # Solve
    sol = solve_ivp(
        rhs_unified,
        t_span,
        U0,
        method="DOP853",
        t_eval=t_eval,
        dense_output=True,
        rtol=rtol,
        atol=atol,
        args=(r_ion_eq,)
    )

    print("Success?", sol.success)
    print("Message:", sol.message)

    if sol.t.size > 0:
        print("Final time:", sol.t[-1])

    # Extract solution
    idx0 = 0
    idx1 = idx0 + 3
    idx2 = idx1 + 3 * n_dm
    idx3 = idx2 + 3
    idx4 = idx3 + 3 * n_dm

    x_ion_sol = sol.y[idx0:idx1]
    x_dms_flat_sol = sol.y[idx1:idx2]

    v_ion_sol = sol.y[idx2:idx3]
    v_dms_flat_sol = sol.y[idx3:idx4]

    # Convert flattened DM solution into shape (n_dm, 3, N)
    n_t = sol.y.shape[1]

    x_dms_sol = x_dms_flat_sol.reshape((n_dm, 3, n_t))
    v_dms_sol = v_dms_flat_sol.reshape((n_dm, 3, n_t))

    return t_eval, x_ion_sol, v_ion_sol, x_dms_sol, v_dms_sol

In [42]:
def analyze_simulation_results(x0_dms, v0_dms, x0_ion, v0_ion, t_span, dt, t_min, m_dm, eps, fit_curve, rutherford, show_plots,
                                rtol=1e-13, atol=1e-16):
    if(rutherford):
        t00, xion00, vion00, xdms00, vdms00 = run_simulation_many_dm_single_ion(x0_dms, v0_dms, x0_ion, v0_ion, t_span, dt, m_dm, eps, run_rutherford=True, rtol=rtol, atol=atol)
    else:
        t00, xion00, vion00, xdms00, vdms00 = run_simulation_many_dm_single_ion(x0_dms, v0_dms, x0_ion, v0_ion, t_span, dt, m_dm, eps, run_rutherford=False, rtol=rtol, atol=atol)
    t0 = t00/um
    xion0 = xion00/um
    vion0 = vion00
    xdms0 = xdms00/um
    vdms0 = vdms00
    x0_ion1, y0_ion1, z0_ion1 = xion0[0], xion0[1], xion0[2] # motion of center ion

    ## Plots 
    if(show_plots):

        # -------------------------
        # Ion coordinates vs time
        # -------------------------
        plt.figure(figsize=(18, 5))

        plt.subplot(1, 3, 1)
        plt.plot(t0, x0_ion1, label="x ion")
        plt.xlabel("t (us)")
        plt.ylabel("x (um)")
        plt.title("Ion x-position vs time")
        plt.legend()

        plt.subplot(1, 3, 2)
        plt.plot(t0, y0_ion1, label="y ion")
        plt.xlabel("t (us)")
        plt.ylabel("y (um)")
        plt.title("Ion y-position vs time")
        plt.legend()

        plt.subplot(1, 3, 3)
        plt.plot(t0, z0_ion1, label="z ion")
        plt.xlabel("t (us)")
        plt.ylabel("z (um)")
        plt.title("Ion z-position vs time")
        plt.legend()

        plt.tight_layout()
        plt.show()

    return t00, xion00, vion00, xdms0, vdms0

In [43]:
def sample_maxwell_boltzmann(n, T, m, plot, seed, bins=100):
    """
    Sample n velocities from a Maxwell-Boltzmann distribution at temperature T for particles of mass m.

    Parameters
    ----------
    n : int
        Number of samples to draw.
    T : float
        Temperature of the distribution (Kelvin).
    m : float
        Mass of the particles.
    seed : int
        Seed for random number generator.

    Returns
    -------
    v : array_like
        Sampled velocities.
    """
    k_B = 1.380649e-23  # Boltzmann constant in J/K
    sigma = np.sqrt(k_B * T / m)
    rng = np.random.default_rng(seed)
    velocities = rng.normal(0, sigma, size=(n, 3))
    speeds = np.linalg.norm(velocities, axis=1)

    if plot:
        v_max = max(np.max(speeds), 5 * sigma)

        bin_edges = np.linspace(0.0, v_max, bins + 1)
        bin_width = bin_edges[1] - bin_edges[0]

        v_grid = np.linspace(0.0, v_max, 1000)

        mb_pdf = (
            4.0 * np.pi * v_grid**2
            * (m / (2.0 * np.pi * k_B * T))**1.5
            * np.exp(-m * v_grid**2 / (2.0 * k_B * T))
        )

        # Convert probability density to approximate probability per bin.
        mb_prob_per_bin = mb_pdf * bin_width

        weights = np.ones_like(speeds) / speeds.size

        plt.figure(figsize=(7, 5))

        plt.hist(
            speeds,
            bins=bin_edges,
            weights=weights,
            alpha=0.5,
            label="Sampled fraction per bin"
        )

        plt.plot(
            v_grid,
            mb_prob_per_bin,
            linewidth=2,
            label="Maxwell-Boltzmann prediction per bin"
        )

        plt.xlabel("Speed [m/s]")
        plt.ylabel("Proportion of particles")
        plt.title("Maxwell-Boltzmann speed distribution")
        plt.legend()
        plt.tight_layout()
        plt.show()
    
    return velocities, speeds

In [44]:
def initial_dm_positions(n_dm, radius, center=(0.0, 0.0, 0.0)):
    """
    Generate initial positions for n_dm dark matter particles evenly distributed
    on the surface of a sphere.

    Parameters
    ----------
    n_dm : int
        Number of dark matter particles.

    radius : float
        Sphere radius in meters.

    center : array-like, shape (3,)
        Sphere center in meters.

    Returns
    -------
    x0_dms : ndarray, shape (n_dm, 3)
        Initial DM positions on the sphere surface.
    """

    center = np.asarray(center, dtype=float)

    if n_dm <= 0:
        raise ValueError("n_dm must be positive")

    indices = np.arange(n_dm)

    golden_angle = np.pi * (3.0 - np.sqrt(5.0))

    z = 1.0 - 2.0 * (indices + 0.5) / n_dm
    r_xy = np.sqrt(1.0 - z**2)

    theta = golden_angle * indices

    x = r_xy * np.cos(theta)
    y = r_xy * np.sin(theta)

    points = np.column_stack([x, y, z])

    x0_dms = center + radius * points

    return x0_dms

# Run Simulation

In [46]:
# DM properties (input parameters)
m_dm = 1e-27  # Mass of DM particle in kg
eps = 1  # Charge of DM particle in units of elementary charge
n_arr = [100,300]  # Number of DM particles
T = 300  # Temperature in Kelvin
R = 30e-6  # Initial distance of DM particles

seeds = [1,2,3,4,5] # Seeds for random number generator
results = {} 

for n in n_arr:
    E_list = []

    for seed in seeds:
        # Initial conditions for the simulation
        x0_ion = np.array([0.0, 0.0, 0.0])  # Initial ion position at the origin
        v0_ion = np.array([0.0, 0.0, 0.0])  # Initial ion velocity at rest
        x0_dms = initial_dm_positions(n_dm=n, radius=R, center=x0_ion)
        v0_dms, speeds = sample_maxwell_boltzmann(n=n, T=T, m=m_dm, plot=False, seed=seed)  # Sample DM velocities at 300 K

        # Run simulation
        t_span=(0, 5e-6)
        dt=1e-9
        t_min = 0
        t_eval, x_ion, v_ion, x_dms, v_dms = analyze_simulation_results(x0_dms, v0_dms, x0_ion, v0_ion, t_span=t_span, dt=dt, t_min=t_min, m_dm=m_dm, eps=eps, fit_curve=False, rutherford=False, show_plots=False)

        # Calculate energy deposited in ion
        E_ion = 0.5 * m_ion * np.linalg.norm(v_ion, axis=0)**2 + 0.5 * m_ion * np.linalg.norm(omega_vec[:, None] * x_ion, axis=0)**2
        E_list.append(E_ion[-1])
    
    results[n] = {
        "E_mean": np.mean(E_list),
        "E_std": np.std(E_list, ddof=1),
        "values": E_list
    }

    print(
        f"n = {n:5d}, "
        f"E = {results[n]['E_mean']:.3e} ± {results[n]['E_std']:.3e} J"
    )

Number of DM particles: 100
Initial ion position: [0. 0. 0.]
Initial ion velocity: [0. 0. 0.]
Success? True
Message: The solver successfully reached the end of the integration interval.
Final time: 4.999e-06
Number of DM particles: 100
Initial ion position: [0. 0. 0.]
Initial ion velocity: [0. 0. 0.]
Success? True
Message: The solver successfully reached the end of the integration interval.
Final time: 4.999e-06
Number of DM particles: 100
Initial ion position: [0. 0. 0.]
Initial ion velocity: [0. 0. 0.]
Success? True
Message: The solver successfully reached the end of the integration interval.
Final time: 4.999e-06
Number of DM particles: 100
Initial ion position: [0. 0. 0.]
Initial ion velocity: [0. 0. 0.]
Success? True
Message: The solver successfully reached the end of the integration interval.
Final time: 4.999e-06
Number of DM particles: 100
Initial ion position: [0. 0. 0.]
Initial ion velocity: [0. 0. 0.]
Success? True
Message: The solver successfully reached the end of the inte